In [1]:
import re
from pathlib import Path
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path("part3_a_results")

JOB_COLORS = {
    "barnes": "#b7d7d8",
    "blackscholes": "#d9c400",
    "canneal": "#d8d0aa",
    "freqmine": "#00c900",
    "radix": "#00d1c1",
    "streamcluster": "#d7b5d8",
    "vips": "#c00000",
}

def parse_k8s_time(s):
    # Example: Sun, 10 May 2026 04:38:38 +0200
    return datetime.strptime(s.strip(), "%a, %d %b %Y %H:%M:%S %z")

def parse_core_range(core_str):
    cores = []
    for part in core_str.split(","):
        part = part.strip()
        if "-" in part:
            a, b = part.split("-")
            cores.extend(range(int(a), int(b) + 1))
        else:
            cores.append(int(part))
    return cores

def parse_jobs_describe(path):
    text = Path(path).read_text()
    chunks = re.split(r"\n(?=Name:\s+parsec-)", text)

    jobs = []

    for chunk in chunks:
        name_match = re.search(r"Name:\s+parsec-([a-zA-Z0-9_-]+)", chunk)
        if not name_match:
            continue

        job = name_match.group(1)

        start_match = re.search(r"Start Time:\s+(.*)", chunk)
        end_match = re.search(r"Completed At:\s+(.*)", chunk)
        node_match = re.search(r"cca-project-nodetype=([a-zA-Z0-9_-]+)", chunk)
        taskset_match = re.search(r"taskset\s+-c\s+([0-9,\-]+)", chunk)

        if not (start_match and end_match and node_match and taskset_match):
            print(f"Warning: missing field for {job}")
            continue

        start = parse_k8s_time(start_match.group(1))
        end = parse_k8s_time(end_match.group(1))
        machine = node_match.group(1)
        cores = parse_core_range(taskset_match.group(1))

        jobs.append({
            "job": job,
            "start": start,
            "end": end,
            "machine": machine,
            "cores": cores,
            "duration_s": (end - start).total_seconds()
        })

    return pd.DataFrame(jobs)

def parse_mcperf(path):
    rows = []
    header = None

    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            if line.startswith("#type"):
                header = line[1:].split()
                continue

            if line.startswith("#"):
                continue

            parts = line.split()
            if header and len(parts) == len(header):
                rows.append(parts)

    df = pd.DataFrame(rows, columns=header)

    numeric_cols = [c for c in df.columns if c != "type"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c])

    # Convert timestamps from ms to seconds
    df["ts_start_s"] = df["ts_start"] / 1000
    df["ts_end_s"] = df["ts_end"] / 1000

    # Convert p95 from microseconds to milliseconds
    df["p95_ms"] = df["p95"] / 1000

    return df

def plot_run(run_id):
    run_dir = BASE_DIR / f"run_{run_id}"
    mcperf_path = run_dir / f"mcperf_{run_id}.txt"
    jobs_path = run_dir / "jobs_describe_final.txt"

    jobs_df = parse_jobs_describe(jobs_path)
    mcperf_df = parse_mcperf(mcperf_path)

    first_start = jobs_df["start"].min()
    last_end = jobs_df["end"].max()

    first_start_epoch = first_start.timestamp()
    last_end_epoch = last_end.timestamp()

    # Keep only mcperf intervals overlapping the batch window
    window_df = mcperf_df[
        (mcperf_df["ts_end_s"] >= first_start_epoch) &
        (mcperf_df["ts_start_s"] <= last_end_epoch)
    ].copy()

    window_df["x_start"] = window_df["ts_start_s"] - first_start_epoch
    window_df["width"] = window_df["ts_end_s"] - window_df["ts_start_s"]

    jobs_df["x_start"] = jobs_df["start"].apply(lambda t: t.timestamp() - first_start_epoch)
    jobs_df["x_end"] = jobs_df["end"].apply(lambda t: t.timestamp() - first_start_epoch)
    jobs_df["width"] = jobs_df["x_end"] - jobs_df["x_start"]

    violations = (window_df["p95_ms"] > 1.0).sum()
    total_points = len(window_df)
    slo_ratio = violations / total_points if total_points > 0 else 0

    makespan = (last_end - first_start).total_seconds()

    print(f"\nRun {run_id}")
    print(f"First job start: {first_start}")
    print(f"Last job end:    {last_end}")
    print(f"Makespan:        {makespan:.2f} s")
    print(f"Datapoints:      {total_points}")
    print(f"Violations:      {violations}")
    print(f"SLO ratio:       {slo_ratio:.4f}")

    fig, (ax1, ax2) = plt.subplots(
        2, 1,
        figsize=(15, 8),
        gridspec_kw={"height_ratios": [2, 1.6]},
        sharex=True
    )

    # Memcached p95 bars
    ax1.bar(
        window_df["x_start"],
        window_df["p95_ms"],
        width=window_df["width"],
        align="edge",
        edgecolor="black",
        linewidth=0.3
    )

    ax1.axhline(1.0, linestyle="--", linewidth=1.5)
    ax1.set_ylabel("p95 latency [ms]")
    ax1.set_title(f"Run {run_id}: memcached p95 latency and batch job placement")
    ax1.grid(axis="y", alpha=0.3)

    # Core timeline
    timeline_rows = []

    for _, row in jobs_df.iterrows():
        for core in row["cores"]:
            timeline_rows.append({
                "machine_core": f"{row['machine']} core {core}",
                "machine": row["machine"],
                "core": core,
                "job": row["job"],
                "x_start": row["x_start"],
                "width": row["width"]
            })

    timeline_df = pd.DataFrame(timeline_rows)

    timeline_df = timeline_df.sort_values(["machine", "core"])
    y_labels = list(timeline_df["machine_core"].drop_duplicates())
    y_map = {label: i for i, label in enumerate(y_labels)}

    for _, row in timeline_df.iterrows():
        y = y_map[row["machine_core"]]
        color = JOB_COLORS.get(row["job"], "gray")

        ax2.barh(
            y=y,
            width=row["width"],
            left=row["x_start"],
            height=0.75,
            color=color,
            edgecolor="black"
        )

        ax2.text(
            row["x_start"] + row["width"] / 2,
            y,
            row["job"],
            ha="center",
            va="center",
            fontsize=8,
            color="black"
        )

    ax2.set_yticks(range(len(y_labels)))
    ax2.set_yticklabels(y_labels)
    ax2.set_xlabel("Time since first batch job start [s]")
    ax2.set_ylabel("Machine / Core")
    ax2.grid(axis="x", alpha=0.3)

    plt.tight_layout()

    out_path = run_dir / f"part3a_run{run_id}_plot.png"
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()

    return {
        "run": run_id,
        "first_job_start": first_start,
        "last_job_end": last_end,
        "makespan_s": makespan,
        "datapoints": total_points,
        "violations": violations,
        "slo_ratio": slo_ratio
    }

summary = []

for run_id in [1, 2, 3]:
    summary.append(plot_run(run_id))

summary_df = pd.DataFrame(summary)
summary_df

FileNotFoundError: [Errno 2] No such file or directory: 'part3_a_results/run_1/jobs_describe_final.txt'